### Session 2: Python Hands-On - Your First AI Function Call
#### Step1

In [2]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.1 MB/s eta 0:00:00


In [5]:
import os
from groq import Groq
from google.colab import userdata

# Retrieve the API key from Colab secrets
api_key = userdata.get("gsk_WORKSHOP_KEY")

client = Groq(api_key=api_key)

def ask_llm(question):
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": question}]
        )
        return response.choices[0].message.content
    except Exception as e:
        # This prints the FULL error details
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {str(e)}")

        # If it's an API error, get more details
        if hasattr(e, 'response'):
            print(f"Status Code: {e.response.status_code}")
            print(f"Response Body: {e.response.text}")
        return None

# Test
result = ask_llm("Whats the capital of France ?'")
if result:
    print(f"SUCCESS: {result}")

SUCCESS: The capital of France is Paris.


#### Step 2

In [6]:
# Test
result = ask_llm("Whats the Weather in Bangalore ?'")
if result:
    print(f"SUCCESS: {result}")

SUCCESS: I'm not currently able to access weather information. However, I can suggest some ways for you to find out the current weather in Bangalore.

1. Check online weather websites: You can check websites like AccuWeather, Weather.com, or the Indian Meteorological Department (IMD) website for the current weather conditions in Bangalore.
2. Use a weather app: You can download a weather app on your smartphone, such as Dark Sky or Weather Underground, to get the current weather conditions in Bangalore.
3. Check social media: Many meteorological departments and weather-related accounts on social media platforms like Twitter or Facebook share live updates on the weather.

Please note that the weather in Bangalore can change frequently, so it's always best to check for the latest updates.

Is there anything else you'd like to know about Bangalore's weather? I'd be happy to help.


#### Giving the Agent Hands
Introducing the Function Definition (JSON Schema).
"This is the instruction manual we give the LLM so it knows what buttons it can push."

In [7]:
import json

# 1. Define the Python Tool
def get_weather(city: str):
    # Mock database for workshop
    mock_db = {"mumbai": "32C, Humid", "delhi": "28C, Smoggy", "bangalore": "27C, Sultry"}
    return mock_db.get(city.lower(), "Data not available")

# 2. Write the TOOL SCHEMA (This is the tricky part students learn)
tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# 3. The AGENTIC CALL
def agent_ask(user_query):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": user_query}],
        tools=tool_schema, # <--- THE MAGIC LINE
        tool_choice="auto"
    )
    return response

# Test it
response = agent_ask("What's the weather like in Bangalore?")
print(response.choices[0].message)
# Output shows: tool_calls with arguments {"city": "Delhi"}

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='p2deksn0w', function=Function(arguments='{"city":"Bangalore"}', name='get_weather'), type='function')])


#### What You Expected (Wrong Mental Model):

User: "Weather in Delhi?"

  ↓
  
LLM: *magically runs Python function*

  ↓
  
LLM: "28C, Smoggy"

#### What Actually Happens (The Agentic Pattern):
User: "Weather in Delhi?"

  ↓
  
LLM: "I need to call the 'get_weather' tool with {'city': 'Delhi'}"

  ↓
  
[LLM STOPS HERE and returns the INTENT to call a tool]

  ↓
  
Python: "Oh! The LLM wants me to run get_weather('Delhi')"

  ↓
  
Python: *actually executes* get_weather('Delhi') → "28C, Smoggy"

  ↓
  
Python: Sends "28C, Smoggy" BACK to LLM

  ↓

  
LLM: "The weather in Delhi is 28C and Smoggy"

### Step 3

In [8]:


# 3. The AGENTIC CALL (Python Executes and Feeds Back (What's Missing)
def agent_ask(user_query):
    print(f"👤 User: {user_query}")

    # STEP 1: Ask LLM (it may request a tool)
    messages = [{"role": "user", "content": user_query}]

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        tools=tool_schema,
        tool_choice="auto"
    )

    response_message = response.choices[0].message
    messages.append(response_message)

    # STEP 2: Check if LLM wants to call a tool
    if response_message.tool_calls:
        print(f"🔧 LLM wants to call: {response_message.tool_calls[0].function.name}")

        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            # STEP 3: Execute the tool
            if function_name == "get_weather":
                city = function_args.get("city")
                result = get_weather(city)
                print(f"📊 Tool result: {result}")

                # STEP 4: Feed result back to LLM
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

        # STEP 5: Get final response from LLM
        final_response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages
        )

        final_answer = final_response.choices[0].message.content
        print(f"🤖 Agent: {final_answer}")
        return final_answer
    else:
        # No tool needed - direct answer
        print(f"🤖 Agent: {response_message.content}")
        return response_message.content

# Test it
agent_ask("What's the weather like in Mumbai?")

👤 User: What's the weather like in Mumbai?
🔧 LLM wants to call: get_weather
📊 Tool result: 32C, Humid
🤖 Agent: However, the actual weather might be different. Unfortunately, I can't access real-time data. 

To get the current weather in Mumbai, you can check a reliable weather website or app.


"However, the actual weather might be different. Unfortunately, I can't access real-time data. \n\nTo get the current weather in Mumbai, you can check a reliable weather website or app."

### Here's the secret: LLMs can't run code. They can only write JSON.

When the LLM returns tool_calls, it's like it's writing a Post-it Note to Python saying 'Please run this function for me.'

Python reads the note, runs get_weather('Delhi'), and then hands the result back to the LLM saying 'Here's what you asked for.'

Only THEN does the LLM write the final answer.

This is the ENTIRE foundation of Agentic AI. The LLM is the Architect, Python is the Construction Worker."